# Notebook 04 — Model Training, Comparison, and Tuning

**Phase 4 learning checkpoint.** We have cleaned, engineered data. Now we *teach a model to predict prices*.

## What you will do here

1. Load the engineered Ames data, define `X` and `y`, and create a held-out test set.
2. Train **three models** with default hyperparameters: Ridge regression, Random Forest, XGBoost.
3. Compare them honestly with **5-fold cross-validation** on the training set.
4. Take the winner and **tune** it with randomized search.
5. Evaluate the tuned model on the held-out test set — the *real* score.
6. Plot predictions vs. actuals and residuals.
7. Save the final model with `joblib` for re-use in later phases.

## Reading order

Pair this notebook with:
- [`docs/06_machine_learning_basics.md`](../docs/06_machine_learning_basics.md) — supervised learning, train/test, Pipelines
- [`docs/07_model_selection.md`](../docs/07_model_selection.md) — Ridge vs. Random Forest vs. XGBoost
- [`docs/08_evaluation_metrics.md`](../docs/08_evaluation_metrics.md) — RMSE, MAE, R²
- [`docs/09_hyperparameter_tuning.md`](../docs/09_hyperparameter_tuning.md) — grid vs. random search, CV

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

from src.data_loader import load_ames, load_zillow_zhvi
from src.features import prepare_modeling_data
from src import models as M

print("Setup complete.")

## 1. Load data and split

`prepare_modeling_data` does the full pipeline (clean → engineer → split features/target) and returns the **log-transformed** target by default.

In [ ]:
X, y = prepare_modeling_data(load_ames(), zhvi=load_zillow_zhvi())
print(f"X: {X.shape}")
print(f"y: {y.shape}, dtype={y.dtype}, range=[{y.min():.2f}, {y.max():.2f}]")

### Why hold out a test set?

Evaluating a model on the same data it trained on tells you only how well it **memorized**. The honest question is *"how does it perform on data it has never seen?"* — we answer that by holding out 20% of the rows *before* any modeling and only touching them at the very end.

`random_state=42` fixes the split so every notebook run uses the same train/test partition.

In [ ]:
X_train, X_test, y_train, y_test = M.split(X, y, test_size=0.20)
print(f"Train: {X_train.shape},  Test: {X_test.shape}")

## 2. Build the three models

Each builder returns a sklearn `Pipeline`. For Ridge, that pipeline scales features first; for the trees, no scaling needed (trees are scale-invariant).

In [ ]:
ridge   = M.build_ridge_pipeline()
rforest = M.build_random_forest_pipeline()
xgb     = M.build_xgboost_pipeline()

print("Ridge:", ridge)
print("\nRandom Forest:", rforest)
print("\nXGBoost:", xgb)

## 3. Quick fit + test-set peek (with caveat!)

Before doing proper cross-validation, let's just fit each model on the training set and score it on the test set. This is the *quickest* but *least reliable* check. Use it for a smell test, not for decisions.

> ⚠️ **Pitfall.** Repeatedly checking the test set, comparing models on it, and picking the winner is a form of *test-set leakage*. The honest workflow: only look at the test score for the **final** model. Until then, use cross-validation.

In [ ]:
# Fit each model once.
ridge.fit(X_train, y_train)
rforest.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# Evaluate against test set in both log and dollar space.
rows = []
for name, m in [("ridge", ridge), ("random_forest", rforest), ("xgboost", xgb)]:
    row = {"model": name, **M.evaluate(m, X_test, y_test)}
    rows.append(row)

quick = pd.DataFrame(rows).set_index("model")
quick.round(4)

## 4. The honest comparison — 5-fold cross-validation

Cross-validation slices the **training** set into 5 folds, trains on 4, tests on 1, rotates, averages. This gives a robust estimate of out-of-sample performance using only training data. The test set stays sacred.

Watch the `rmse_std` column too — a model that scores well *and* stably across folds is more trustworthy than one with the same mean but a wider spread.

In [ ]:
models_dict = {
    "ridge":         M.build_ridge_pipeline(),
    "random_forest": M.build_random_forest_pipeline(),
    "xgboost":       M.build_xgboost_pipeline(),
}
comparison = M.compare_cv(models_dict, X_train, y_train, cv=5)
comparison.round(4)

The metrics are computed on the log-transformed target. RMSE values around 0.12–0.14 in log-space correspond to roughly 12–14% typical error in dollar-space. R² near 0.90 means the model explains ~90% of the variation in (log) sale prices.

**Per-fold detail** for a single model — useful for spotting whether one fold is a wild outlier:

In [ ]:
folds = M.cv_score(M.build_xgboost_pipeline(), X_train, y_train, cv=5)
folds.round(4)

## 5. Tune the winner — XGBoost

XGBoost has the most hyperparameter knobs of the three. `tune_xgboost` runs `RandomizedSearchCV` over a sensible 6-dimensional space with 30 random combinations and 5-fold CV — so 150 model fits in total.

This will take about a minute on a modern laptop.

> 📘 **Why random search beats grid search.** Most of the dimensions in a hyperparameter space don't matter equally. Grid search wastes evaluations exploring all combinations of unimportant dimensions; random search samples them lightly and covers the important ones better. See [doc 09](../docs/09_hyperparameter_tuning.md) for the full argument.

In [ ]:
best_model, best_params, cv_results = M.tune_xgboost(
    X_train, y_train,
    n_iter=30,
    cv=5,
)

print("Best params:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

print("\nTop 5 configurations:")
cv_results.head(5).round(4)

## 6. Final test-set evaluation

Now — and only now — we look at the held-out test set with the tuned model. The numbers here are our **honest** estimate of real-world performance.

In [ ]:
final = M.evaluate(best_model, X_test, y_test, target_is_log=True)
print("FINAL TEST-SET METRICS (tuned XGBoost):")
for k, v in final.items():
    # Only RMSE and MAE are in dollars; R^2 is always unitless.
    if k.startswith("dollar_") and not k.endswith("_r2"):
        print(f"  {k:14s}: ${v:,.0f}")
    else:
        print(f"  {k:14s}: {v:.4f}")

Read this as: "Our model's typical prediction error on a held-out Ames home is roughly **$ that dollar_mae number**, and the model explains **~95%** of the variance in log SalePrice."

Note: dollar-space RMSE is bigger than dollar-space MAE because RMSE penalizes the model's *worst* mistakes (often on the most expensive homes) more heavily. See [doc 08](../docs/08_evaluation_metrics.md) for when to use each.

## 7. Diagnostic plots

### Predictions vs. actuals

Each dot is one home. A perfect model would put every dot on the red diagonal. We plot in **dollar space** so the axes are interpretable.

In [ ]:
y_pred_log = best_model.predict(X_test)
y_pred_dollars = np.expm1(y_pred_log)
y_true_dollars = np.expm1(y_test.to_numpy())

M.plot_predictions_vs_actuals(
    y_true_dollars, y_pred_dollars,
    title="Tuned XGBoost — test set (USD)",
    label_xy=("actual saleprice ($)", "predicted saleprice ($)"),
)

### Residuals

Residuals = `y_true - y_pred`. For a well-fit model this should look approximately normal, centered near zero, with no obvious skew. A leftward tail means the model systematically under-predicts on expensive homes; a rightward tail is the opposite.

In [ ]:
M.plot_residuals(
    y_test.to_numpy(),
    y_pred_log,
    title="Residuals in log space",
)

## 8. Save the final model

We persist the fitted Pipeline (preprocessing + tuned XGBoost) so later phases (interpretability, ROI, dashboard) can load it without retraining.

In [ ]:
saved_path = M.save_model(best_model, name="xgb_ames_tuned")
print(f"Saved: {saved_path}")

# Verify we can load it back and reproduce predictions.
loaded = M.load_model("xgb_ames_tuned")
y_pred_again = loaded.predict(X_test)
print("Reload predictions match originals:", np.allclose(y_pred_again, y_pred_log))

## 9. Wrap-up — what did you learn?

Write your answers down before moving on:

1. **Why the test set is sacred.** What's the difference between picking a model based on test scores you've already peeked at, vs. picking a model based on cross-validation? Why is the second one more honest?
2. **Reading the comparison table.** In the 5-fold CV results, look at both `rmse_mean` and `rmse_std`. Which model wins on mean? Which is most *stable* across folds? Why might that distinction matter?
3. **Linear vs. trees.** Ridge regression with engineered features held up surprisingly well. What does that tell you about how good the Phase 3 feature engineering was?
4. **The tuning lift.** How much did the tuned XGBoost improve over the default? Was the gain worth the compute?
5. **Dollar vs. log RMSE.** Why is the test-set `dollar_rmse` larger than `dollar_mae`? What kind of mistake is RMSE penalizing extra?
6. **Residuals plot.** Does the residual distribution look symmetric and centered? If not, in which direction is the model biased?
7. **What you'd try next.** Suggest one thing you'd try in Phase 5 / 6 that *might* improve this model — and one thing you'd refuse to do (and why).

When you can answer these, you're ready for **Phase 5: Time-Series Forecasting**.